In [0]:
from pyspark.sql.functions import col
import builtins
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime

source_table = "healthcare_migration.bronze.claims"
target_table = "healthcare_migration.silver.claims_cleaned"
validation_report_table = "healthcare_migration.gold.validation_report"
audit_log_table = "healthcare_migration.gold.audit_log"
source_df = spark.read.table(source_table)
target_df = spark.read.table(target_table)

print("Tables loaded successfully")

source_count = source_df.count()

target_count = target_df.count()

row_count_status = (
    "PASS"
    if source_count == target_count
    else "FAIL"
)

null_count = target_df.filter(
    col("PATIENTID").isNull()
).count()

null_threshold = 1000

null_status = (
    "PASS"
    if null_count <= null_threshold
    else "FAIL"
)

duplicate_count = (
    target_df.groupBy("Id")
    .count()
    .filter(col("count") > 1)
    .count()
)

duplicate_status = (
    "PASS"
    if duplicate_count == 0
    else "FAIL"
)

source_columns = set(source_df.columns)

target_columns = set(target_df.columns)

schema_status = (
    "PASS"
    if source_columns.issubset(target_columns)
    else "FAIL"
)

reconciliation_difference = (
    int(source_count) - int(target_count)
)

reconciliation_difference = builtins.abs(
    reconciliation_difference
)

reconciliation_status = (
    "PASS"
    if reconciliation_difference == 0
    else "FAIL"
)

validation_results = [

    (
        "Row Count Validation",
        row_count_status,
        source_count,
        target_count
    ),

    (
        "Null Validation",
        null_status,
        null_count,
        null_threshold
    ),

    (
        "Duplicate Validation",
        duplicate_status,
        duplicate_count,
        0
    ),

    (
        "Schema Validation",
        schema_status,
        len(source_columns),
        len(target_columns)
    ),

    (
        "Reconciliation Validation",
        reconciliation_status,
        reconciliation_difference,
        0
    )

]

validation_schema = StructType([

    StructField(
        "Validation_Name",
        StringType(),
        True
    ),

    StructField(
        "Status",
        StringType(),
        True
    ),

    StructField(
        "Actual_Value",
        LongType(),
        True
    ),

    StructField(
        "Expected_Value",
        LongType(),
        True
    )

])

validation_df = spark.createDataFrame(
    validation_results,
    validation_schema
)

display(validation_df)

spark.sql(f"DROP TABLE IF EXISTS {validation_report_table}")

validation_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(validation_report_table)

print("Validation report saved successfully")

audit_data = [

    (
        source_table,
        target_table,
        source_count,
        target_count,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    )

]

audit_schema = StructType([

    StructField(
        "Source_Table",
        StringType(),
        True
    ),

    StructField(
        "Target_Table",
        StringType(),
        True
    ),

    StructField(
        "Source_Count",
        LongType(),
        True
    ),

    StructField(
        "Target_Count",
        LongType(),
        True
    ),

    StructField(
        "Load_Timestamp",
        StringType(),
        True
    )

])

audit_df = spark.createDataFrame(
    audit_data,
    audit_schema
)

display(audit_df)

spark.sql(f"DROP TABLE IF EXISTS {audit_log_table}")

audit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(audit_log_table)

print("Audit log saved successfully")

print("===================================")
print("MIGRATION VALIDATION COMPLETED")
print("===================================")

Tables loaded successfully


Validation_Name,Status,Actual_Value,Expected_Value
Row Count Validation,PASS,49339,49339
Null Validation,FAIL,35163,1000
Duplicate Validation,FAIL,1,0
Schema Validation,PASS,84,85
Reconciliation Validation,PASS,0,0


Validation report saved successfully


Source_Table,Target_Table,Source_Count,Target_Count,Load_Timestamp
healthcare_migration.bronze.claims,healthcare_migration.silver.claims_cleaned,49339,49339,2026-05-09 05:42:08


Audit log saved successfully
MIGRATION VALIDATION COMPLETED
